In [5]:
import pandas as pd
import numpy as np
from datetime import timedelta

class TMIndicator:

    def __init__(self,df: pd.DataFrame) -> None:
        self.df = df.copy()
        # Define the parameters
        self.half_length = 10
        self.price_column = 'close'
        self.bands_deviations = 2.4
        self.koeff = 0.0001
        self.interpolate = True
        # Initialize buffers
        self.tm_buffer = np.zeros(len(self.df))
        self.up_buffer = np.zeros(len(self.df))
        self.dn_buffer = np.zeros(len(self.df))
        self.wu_buffer = np.zeros(len(self.df))
        self.wd_buffer = np.zeros(len(self.df))
        self.up_arrow = np.full(len(self.df), np.nan)
        self.dn_arrow = np.full(len(self.df), np.nan)

    # Calculate the TMA and bands
    def calculate_tma(self, half_length, price_column, bands_deviations, koeff):
        full_length = 2.0 * half_length + 1.0
        
        for i in range(len(self.df)):
            sum_val = (half_length + 1) * self.df[price_column].iloc[i]
            sumw = half_length + 1
            for j in range(1, half_length + 1):
                if i + j < len(self.df):
                    sum_val += (half_length - j + 1) * self.df[price_column].iloc[i + j]
                    sumw += (half_length - j + 1)
                if i - j >= 0:
                    sum_val += (half_length - j + 1) * self.df[price_column].iloc[i - j]
                    sumw += (half_length - j + 1)
            
            self.tm_buffer[i] = sum_val / sumw
            
            if i >= half_length:
                diff = self.df[price_column].iloc[i] - self.tm_buffer[i]
                if i == half_length:
                    self.wu_buffer[i] = np.power(diff, 2) if diff >= 0 else 0
                    self.wd_buffer[i] = np.power(diff, 2) if diff < 0 else 0
                else:
                    self.wu_buffer[i] = (self.wu_buffer[i-1] * (full_length - 1) + np.power(diff, 2)) / full_length if diff >= 0 else self.wu_buffer[i-1] * (full_length - 1) / full_length
                    self.wd_buffer[i] = (self.wd_buffer[i-1] * (full_length - 1) + np.power(diff, 2)) / full_length if diff < 0 else self.wd_buffer[i-1] * (full_length - 1) / full_length

                self.up_buffer[i] = self.tm_buffer[i] + bands_deviations * np.sqrt(self.wu_buffer[i])
                self.dn_buffer[i] = self.tm_buffer[i] - bands_deviations * np.sqrt(self.wd_buffer[i])
        
        return self.tm_buffer, self.up_buffer, self.dn_buffer

    def calculate(self):
        return self.calculate_tma(self.half_length, self.price_column, self.bands_deviations, self.koeff)
    def mainloop(self):
        # Generate arrows
        self.tm_buffer, self.up_buffer, self.dn_buffer = self.calculate()
        for i in range(1, len(self.df) - 1):
            if self.df['high'].iloc[i+1] > self.up_buffer[i+1] and self.df['close'].iloc[i+1] > self.df['open'].iloc[i+1] and self.df['close'].iloc[i] < self.df['open'].iloc[i]:
                self.up_arrow[i] = self.df['high'].iloc[i] + self.df['close'].rolling(window=20).mean().iloc[i] + self.koeff
            if self.df['low'].iloc[i+1] < self.dn_buffer[i+1] and self.df['close'].iloc[i+1] < self.df['open'].iloc[i+1] and self.df['close'].iloc[i] > self.df['open'].iloc[i]:
                self.dn_arrow[i] = self.df['low'].iloc[i] - self.df['close'].rolling(window=20).mean().iloc[i] - self.koeff
        # decimal_length = self.df['open'].iloc[0].astype(str).split('.')[1].__len__()
        # self.df.loc[:, 'tm_buffer'] = self.tm_buffer.round(decimal_length)
        # self.df.loc[:, 'up_buffer'] = self.up_buffer.round(decimal_length)
        # self.df.loc[:, 'dn_buffer'] = self.dn_buffer.round(decimal_length)
        # self.df.loc[:, 'up_arrow'] = self.up_arrow.round(decimal_length)
        # self.df.loc[:, 'dn_arrow'] = self.dn_arrow.round(decimal_length)
        # self.df.loc[:, 'SELL_TM'] = self.df.apply(lambda row: min(row['open'], row['close']) <= row['up_buffer'] <= max(row['open'], row['close']), axis=1)
        # self.df.loc[:, 'BUY_TM'] = self.df.apply(lambda row: min(row['open'], row['close']) <= row['dn_buffer'] <= max(row['open'], row['close']), axis=1)
        # return self.df
        decimal_length = len(str(self.df['open'].iloc[0]).split('.')[1])
        
        # Use .loc to avoid the SettingWithCopyWarning
        self.df.loc[:, 'tm_buffer'] = np.round(self.tm_buffer, decimal_length)
        self.df.loc[:, 'up_buffer'] = np.round(self.up_buffer, decimal_length)
        self.df.loc[:, 'dn_buffer'] = np.round(self.dn_buffer, decimal_length)
        self.df.loc[:, 'up_arrow'] = np.round(self.up_arrow, decimal_length)
        self.df.loc[:, 'dn_arrow'] = np.round(self.dn_arrow, decimal_length)
        
        # Vectorized computation for SELL_TM and BUY_TM
        self.df.loc[:, 'SELL_TM'] = (self.df[['open', 'close']].min(axis=1) <= self.df['up_buffer']) & (self.df['up_buffer'] <= self.df[['open', 'close']].max(axis=1))
        self.df.loc[:, 'BUY_TM'] = (self.df[['open', 'close']].min(axis=1) <= self.df['dn_buffer']) & (self.df['dn_buffer'] <= self.df[['open', 'close']].max(axis=1))
        
        return self.df

class ExtremeSpike:
    def __init__(self,df: pd.DataFrame,assest:str) -> None:
        self.df = df.copy()
        # self.df['datetime'] = pd.to_datetime(self.df['datetime'])
        # self.df['UTC'] = self.df['datetime'] + timedelta(hours=5)

        # # Calculate GMT (UTC + 2 hours)
        # self.df['GMT'] = self.df['UTC'] + timedelta(hours=2)
        # Define constants
        self.MINOR_MIN_EXTREME_HEIGHT_ATRS = 2.0
        self.MAJOR_TO_MINOR_HEIGHT_RATIO = 2.5
        self.MINOR_MIN_EXTREME_WIDTH = 2
        self.MAJOR_MIN_EXTREME_WIDTH = 2
        self.RANGE_AVERAGING_PERIOD = 250
        # LINE_VALUE_DOWN = -1.0
        # LINE_VALUE_UP = 1.0
        self.LINE_MINOR = 'minor'
        self.LINE_MAJOR = 'major'
        self.LINE_SHADOW = 'shadow'
        self.LINE_STABLE = 'stable'
        self.NoRepaint = False
        self.LINE_VALUE_UP = 1.0
        self.LINE_VALUE_FLAT = 0.0
        self.LINE_VALUE_DOWN = -1.0
        # Initialize columns for signals and other intermediate calculations
        self.df['ATR'] = self.df['high'] - self.df['low']
        self.df['ATR_SMA'] = self.df['ATR'].rolling(window=self.RANGE_AVERAGING_PERIOD).mean()
        self.df['MinorMinExtremeHeight'] = self.df['ATR_SMA'] * self.MINOR_MIN_EXTREME_HEIGHT_ATRS
        self.df['MajorMinExtremeHeight'] = self.df['MinorMinExtremeHeight'] * self.MAJOR_TO_MINOR_HEIGHT_RATIO
        self.df['line1'] = 0.0
        self.df['line2'] = 0.0
        self.df['line3'] = 0.0
        self.df['line4'] = 0.0
        self.df['line5'] = 0.0
        # Initialize state variables
        self.minor_low_extreme_price = self.df['low'].iloc[0]
        self.minor_hi_extreme_price = self.df['high'].iloc[0]
        self.major_low_extreme_price = self.df['low'].iloc[0]
        self.major_hi_extreme_price = self.df['high'].iloc[0]
        self.minor_low_extreme_idx = 0
        self.minor_hi_extreme_idx = 0
        self.major_low_extreme_idx = 0
        self.major_hi_extreme_idx = 0
        self.minor_extreme_mode = 0
        self.major_extreme_mode = 0
        self.first_minor_low = True
        self.first_minor_high = True
        self.first_major_low = True
        self.first_major_high = True

    def eraseExtreme(self,lineType, barIdx, value):
        drawShadow = (lineType == self.LINE_MAJOR) and (self.NoRepaint or (value == self.LINE_VALUE_UP and self.df['line1'].iloc[barIdx] != 0) or (value == self.LINE_VALUE_DOWN and self.df['line2'].iloc[barIdx] != 0))
        self.drawExtreme(lineType, barIdx, self.LINE_VALUE_FLAT)
        if drawShadow:
            self.draw(self.LINE_SHADOW, barIdx, value)

    def drawExtreme(self,lineType, barIdx, value):
        if not self.NoRepaint:
            self.draw(lineType, barIdx, value)
            self.drawStableLine(lineType, barIdx, value)

    def drawStableLine(self,lineType, barIdx, value):
        if lineType == self.LINE_MAJOR:
            return False
        self.draw(self.LINE_STABLE, barIdx, value)
        return True

    def draw(self, lineType, barIdx, value):
        if lineType == self.LINE_MAJOR:
            self.updateLine('line1', 'line2', barIdx, value)
        elif lineType == self.LINE_MINOR:
            self.updateLine('line5', 'line5', barIdx, value)
        elif lineType == self.LINE_SHADOW:
            self.updateLine('line3', 'line3', barIdx, value)
        elif lineType == self.LINE_STABLE:
            self.updateLine('line4', 'line4', barIdx, value)

    def updateLine(self, lineUp, lineDown, barIdx, value):
        if value in [self.LINE_VALUE_FLAT, self.LINE_VALUE_UP]:
            self.df.loc[barIdx, lineUp] = value
        if value in [self.LINE_VALUE_FLAT, self.LINE_VALUE_DOWN]:
            self.df.loc[barIdx, lineDown] = value

    # Helper functions
    def check_for_extremes(self,low_extreme_idx, low_extreme_price, hi_extreme_idx, hi_extreme_price, first_low, first_high, extreme_mode, min_extreme_height, min_extreme_width, current_idx, low, high, lineType, df):
        signal = 0
        line_value = 0.0
        # Check for Bottom
        if extreme_mode > -1:
            if low < low_extreme_price:
                if not first_low:
                    self.eraseExtreme(lineType, low_extreme_idx, self.LINE_VALUE_DOWN)
                low_extreme_price = low
                low_extreme_idx = current_idx
                first_low = False
            elif low > low_extreme_price:
                self.drawExtreme(lineType, low_extreme_idx, self.LINE_VALUE_DOWN)
                first_low = False
                if ((low - low_extreme_price) >= min_extreme_height) and ((current_idx - low_extreme_idx) >= min_extreme_width):
                    extreme_mode = -1
                    hi_extreme_price = high
                    hi_extreme_idx = current_idx
                    first_high = True
                    first_low = True
                    line_value = self.LINE_VALUE_DOWN
                    if self.NoRepaint:
                        self.draw(lineType, low_extreme_idx, self.LINE_VALUE_DOWN)
                    self.drawStableLine(lineType, low_extreme_idx, self.LINE_VALUE_FLAT)
                    # if lineType == self.LINE_MINOR:
                    #     signal = 1  # Minor buy signal
                    #     df.at[low_extreme_idx, 'line1'] = line_value
                    # else:
                    #     signal = 2  # Major buy signal
                    #     df.at[low_extreme_idx, 'line2'] = line_value

        # Check for Top
        if extreme_mode < 1:
            if high > hi_extreme_price:
                if not first_high:
                    self.eraseExtreme(lineType, hi_extreme_idx, self.LINE_VALUE_UP)
                hi_extreme_price = high
                hi_extreme_idx = current_idx
                first_high = False
            elif high < hi_extreme_price:
                self.drawExtreme(lineType, hi_extreme_idx, self.LINE_VALUE_UP)
                first_high = False
                if ((hi_extreme_price - low) >= min_extreme_height) and ((current_idx - hi_extreme_idx) >= min_extreme_width):
                    extreme_mode = 1
                    low_extreme_price = low
                    low_extreme_idx = current_idx
                    first_high = True
                    first_low = True
                    line_value = self.LINE_VALUE_UP
                    if self.NoRepaint:
                        self.draw(lineType, hi_extreme_idx, self.LINE_VALUE_UP)
                    self.drawStableLine(lineType, hi_extreme_idx, self.LINE_VALUE_FLAT)
                    # if lineType == self.LINE_MINOR:
                    #     signal = -1  # Minor sell signal
                    #     df.at[hi_extreme_idx, 'line1'] = line_value
                    # else:
                    #     signal = -2  # Major sell signal
                    #     df.at[hi_extreme_idx, 'line2'] = line_value

        return low_extreme_idx, hi_extreme_idx, low_extreme_price, hi_extreme_price, first_low, first_high, extreme_mode, signal

    def mainloop(self):
        # Process each row
        for idx in range(1, len(self.df)):
            # Minor extremes

            self.minor_low_extreme_idx, self.minor_hi_extreme_idx, self.minor_low_extreme_price, self.minor_hi_extreme_price, self.first_minor_low, self.first_minor_high, self.minor_extreme_mode, minor_signal = self.check_for_extremes(
                self.minor_low_extreme_idx, self.minor_low_extreme_price,
                self.minor_hi_extreme_idx, self.minor_hi_extreme_price, self.first_minor_low, self.first_minor_high, self.minor_extreme_mode,
                self.df['MinorMinExtremeHeight'].iloc[idx], self.MINOR_MIN_EXTREME_WIDTH, idx, self.df['low'].iloc[idx], self.df['high'].iloc[idx], 'minor', self.df
            )

            # Major extremes

            self.major_low_extreme_idx, self.major_hi_extreme_idx, self.major_low_extreme_price, self.major_hi_extreme_price, self.first_major_low, self.first_major_high, self.major_extreme_mode, major_signal = self.check_for_extremes(
                self.major_low_extreme_idx, self.major_low_extreme_price,
                self.major_hi_extreme_idx, self.major_hi_extreme_price, self.first_major_low, self.first_major_high, self.major_extreme_mode,
                self.df['MajorMinExtremeHeight'].iloc[idx], self.MAJOR_MIN_EXTREME_WIDTH, idx, self.df['low'].iloc[idx], self.df['high'].iloc[idx], 'major', self.df
            )

        return self.df[['line1','line2','line4','line5']]
# class MovAvg:
#     def __init__(self,df):
#         self.df = df

#     # Function to calculate moving averages
#     def calculate_ma(self,series, period, mode):
#         if mode == 0:  # SMA
#             return series.rolling(window=period).mean()
#         elif mode == 1:  # EMA
#             return series.ewm(span=period, adjust=False).mean()
#         elif mode == 2:  # SMMA - Simple Modified Moving Average
#             # This is a placeholder for SMMA calculation
#             return series.ewm(alpha=1.0 / period).mean()
#         elif mode == 3:  # LWMA - Linear Weighted Moving Average
#             weights = np.arange(1, period + 1)
#             return series.rolling(period).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)
#         else:
#             raise ValueError("Invalid mode for moving average")
        
#     def mainloop(self):
#         # Parameters
#         faster_mode = 3  # LWMA
#         faster_ma_period = 3
#         slower_mode = 3  # LWMA
#         slower_ma_period = 3

#         # Calculate moving averages
#         self.df['FasterMA'] = self.calculate_ma(self.df['close'], faster_ma_period, faster_mode)
#         self.df['SlowerMA'] = self.calculate_ma(self.df['open'], slower_ma_period, slower_mode)

#         # Initialize columns for cross signals
#         self.df['CrossUp'] = np.nan
#         self.df['CrossDown'] = np.nan

#         # Calculate range
#         self.df['Range'] = self.df['high'] - self.df['low']
#         self.df['AvgRange'] = self.df['Range'].rolling(window=10).mean()

#         # Detect cross signals
#         for i in range(1, len(self.df)):
#             if (
#                 self.df['FasterMA'].iloc[i] > self.df['SlowerMA'].iloc[i] and
#                 self.df['FasterMA'].iloc[i-1] < self.df['SlowerMA'].iloc[i-1] and
#                 self.df['FasterMA'].iloc[i] > self.df['FasterMA'].iloc[i-1]
#             ):
#                 self.df.at[i, 'CrossUp'] = self.df['low'].iloc[i] - self.df['AvgRange'].iloc[i] * 0.3
#             elif (
#                 self.df['FasterMA'].iloc[i] < self.df['SlowerMA'].iloc[i] and
#                 self.df['FasterMA'].iloc[i-1] > self.df['SlowerMA'].iloc[i-1] and
#                 self.df['FasterMA'].iloc[i] < self.df['FasterMA'].iloc[i-1]
#             ):
#                 self.df.at[i, 'CrossDown'] = self.df['high'].iloc[i] + self.df['AvgRange'].iloc[i] * 0.3
#         return self.df[['CrossDown','CrossUp','FasterMA','SlowerMA']]

class MovAvg:
    def __init__(self, df):
        """
        Initialize the MovAvg class with a DataFrame.
        :param df: pandas DataFrame with columns 'close', 'open', 'high', and 'low'.
        """
        self.df = df

    def calculate_ma(self, series, period, mode):
        """
        Calculate the moving average based on the specified mode.
        :param series: pandas Series to calculate the moving average on.
        :param period: int, the window period for the moving average.
        :param mode: int, the mode of moving average (0: SMA, 1: EMA, 2: SMMA, 3: LWMA).
        :return: pandas Series with the moving average.
        """
        if mode == 0:  # Simple Moving Average (SMA)
            return series.rolling(window=period).mean()
        elif mode == 1:  # Exponential Moving Average (EMA)
            return series.ewm(span=period, adjust=False).mean()
        elif mode == 2:  # Simple Modified Moving Average (SMMA)
            return series.ewm(alpha=1.0 / period).mean()
        elif mode == 3:  # Linear Weighted Moving Average (LWMA)
            weights = np.arange(1, period + 1)
            return series.rolling(period).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)
        else:
            raise ValueError("Invalid mode for moving average")

    def mainloop(self):
        """
        Main loop to calculate moving averages and detect cross signals.
        :return: pandas DataFrame with 'CrossDown', 'CrossUp', 'FasterMA', and 'SlowerMA' columns.
        """
        # Parameters
        faster_mode = 3  # LWMA
        faster_ma_period = 3
        slower_mode = 3  # LWMA
        slower_ma_period = 3

        # Calculate moving averages
        self.df['FasterMA'] = self.calculate_ma(self.df['close'], faster_ma_period, faster_mode)
        self.df['SlowerMA'] = self.calculate_ma(self.df['open'], slower_ma_period, slower_mode)

        # Calculate range and average range
        self.df['Range'] = self.df['high'] - self.df['low']
        self.df['AvgRange'] = self.df['Range'].rolling(window=10).mean()

        # Initialize columns for cross signals
        self.df['CrossUp'] = np.nan
        self.df['CrossDown'] = np.nan

        # Detect cross signals
        crosses_up = (
            (self.df['FasterMA'] > self.df['SlowerMA']) &
            (self.df['FasterMA'].shift(1) < self.df['SlowerMA'].shift(1)) &
            (self.df['FasterMA'] > self.df['FasterMA'].shift(1))
        )
        crosses_down = (
            (self.df['FasterMA'] < self.df['SlowerMA']) &
            (self.df['FasterMA'].shift(1) > self.df['SlowerMA'].shift(1)) &
            (self.df['FasterMA'] < self.df['FasterMA'].shift(1))
        )

        self.df.loc[crosses_up, 'CrossUp'] = self.df['low'] - self.df['AvgRange'] * 0.3
        self.df.loc[crosses_down, 'CrossDown'] = self.df['high'] + self.df['AvgRange'] * 0.3

        return self.df[['CrossDown', 'CrossUp', 'FasterMA', 'SlowerMA']]

# Example usage
# df = pd.read_csv('your_data.csv')  # Assuming the DataFrame is already loaded
# ma = MovAvg(df)
# result = ma.mainloop()
# print(result)


dataframe = pd.read_csv('common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')
extreme_spk = ExtremeSpike(dataframe,assest='EURUSD')
ex_result = extreme_spk.mainloop()
tm_ind = TMIndicator(dataframe)
tm_result = tm_ind.mainloop()
mv_avg = MovAvg(dataframe)
mv_avg_result = mv_avg.mainloop()
new_df = pd.concat([ex_result, tm_result,mv_avg_result],axis=1)
new_df['UTC'] = pd.to_datetime(new_df['datetime']) + timedelta(hours=5)
new_df['GMT'] = new_df['UTC'] + timedelta(hours=2)
new_df = new_df.reindex(columns=['datetime','UTC','GMT', 'open',	'high', 'low', 'close', 'volume', 'tm_buffer','up_buffer','dn_buffer','up_arrow','dn_arrow','line1','line2','line4','line5','SELL_TM','BUY_TM','CrossDown','CrossUp','FasterMA','SlowerMA'])
new_df.to_csv('common/MachineLearningModel/output/outputEurusd.csv')


In [1]:
import pandas as pd
import numpy as np

# Function to calculate moving averages
def calculate_ma(series, period, mode):
    if mode == 0:  # SMA
        return series.rolling(window=period).mean()
    elif mode == 1:  # EMA
        return series.ewm(span=period, adjust=False).mean()
    elif mode == 2:  # SMMA - Simple Modified Moving Average
        # This is a placeholder for SMMA calculation
        return series.ewm(alpha=1.0 / period).mean()
    elif mode == 3:  # LWMA - Linear Weighted Moving Average
        weights = np.arange(1, period + 1)
        return series.rolling(period).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)
    else:
        raise ValueError("Invalid mode for moving average")

# Sample OHLC data (Replace with actual data)
data = pd.DataFrame({
    'Open': [1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0],
    'High': [1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1],
    'Low': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9],
    'Close': [1.15, 1.25, 1.35, 1.45, 1.55, 1.65, 1.75, 1.85, 1.95, 2.05]
})

# Parameters
faster_mode = 3  # LWMA
faster_ma_period = 3
slower_mode = 3  # LWMA
slower_ma_period = 3

# Calculate moving averages
data['FasterMA'] = calculate_ma(data['Close'], faster_ma_period, faster_mode)
data['SlowerMA'] = calculate_ma(data['Open'], slower_ma_period, slower_mode)

# Initialize columns for cross signals
data['CrossUp'] = np.nan
data['CrossDown'] = np.nan

# Calculate range
data['Range'] = data['High'] - data['Low']
data['AvgRange'] = data['Range'].rolling(window=10).mean()

# Detect cross signals
for i in range(1, len(data)):
    if (
        data['FasterMA'].iloc[i] > data['SlowerMA'].iloc[i] and
        data['FasterMA'].iloc[i-1] < data['SlowerMA'].iloc[i-1] and
        data['FasterMA'].iloc[i] > data['FasterMA'].iloc[i-1]
    ):
        data.at[i, 'CrossUp'] = data['Low'].iloc[i] - data['AvgRange'].iloc[i] * 0.3
    elif (
        data['FasterMA'].iloc[i] < data['SlowerMA'].iloc[i] and
        data['FasterMA'].iloc[i-1] > data['SlowerMA'].iloc[i-1] and
        data['FasterMA'].iloc[i] < data['FasterMA'].iloc[i-1]
    ):
        data.at[i, 'CrossDown'] = data['High'].iloc[i] + data['AvgRange'].iloc[i] * 0.3

print(data)


   Open  High  Low  Close  FasterMA  SlowerMA  CrossUp  CrossDown  Range  \
0   1.1   1.2  1.0   1.15       NaN       NaN      NaN        NaN    0.2   
1   1.2   1.3  1.1   1.25       NaN       NaN      NaN        NaN    0.2   
2   1.3   1.4  1.2   1.35  1.283333  1.233333      NaN        NaN    0.2   
3   1.4   1.5  1.3   1.45  1.383333  1.333333      NaN        NaN    0.2   
4   1.5   1.6  1.4   1.55  1.483333  1.433333      NaN        NaN    0.2   
5   1.6   1.7  1.5   1.65  1.583333  1.533333      NaN        NaN    0.2   
6   1.7   1.8  1.6   1.75  1.683333  1.633333      NaN        NaN    0.2   
7   1.8   1.9  1.7   1.85  1.783333  1.733333      NaN        NaN    0.2   
8   1.9   2.0  1.8   1.95  1.883333  1.833333      NaN        NaN    0.2   
9   2.0   2.1  1.9   2.05  1.983333  1.933333      NaN        NaN    0.2   

   AvgRange  
0       NaN  
1       NaN  
2       NaN  
3       NaN  
4       NaN  
5       NaN  
6       NaN  
7       NaN  
8       NaN  
9       0.2  
